# 04. Sleman as the Daytime Engine: Flows Between Regions

**Data sources:** `od_home_visits_kapanewon_monthly.csv`, `od_origin_homes_monthly.csv`, `od_summary.csv`

**Questions**
1. How strongly are the five regencies/city connected? Where do residents of each regency go during the day?
2. Which kapanewon pairs have the largest visitor flows?
3. Which kapanewon attract visitors from the most origins (regional hubs)?

OD values are **distinct visitors per month**, not trips. We use Dec 2021 - Feb 2022 only (README limitation 6).

In [ ]:
# @title 0. Setup (jalankan sekali, otomatis di Google Colab)
# Di Colab: membuat folder kerja, menulis modul helper, dan mengunduh data dari Zenodo.
# Di laptop (repo GitHub): sel ini tidak melakukan apa-apa.
import os, sys, hashlib
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    BASE = "/content/jogjapulse"
    for d in ["src", "data", "figures", "notebooks"]:
        os.makedirs(f"{BASE}/{d}", exist_ok=True)
    open(f"{BASE}/src/__init__.py", "w").close()
    with open(f"{BASE}/src/jogjapulse.py", "w") as f:
        f.write('"""\nShared helpers for the JogjaPulse human mobility project.\n\nAll notebooks import from here so that loading, cleaning and normalisation\nare done the same way everywhere.\n\nDataset: Widyawan et al. (2026). JogjaPulse: An Aggregated Spatio-Temporal\nMobility Dataset for Yogyakarta, Indonesia. Zenodo.\nhttps://doi.org/10.5281/zenodo.23157460  (CC BY 4.0)\n"""\nfrom __future__ import annotations\n\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\n\n# --------------------------------------------------------------------------\n# Paths\n# --------------------------------------------------------------------------\nROOT = Path(__file__).resolve().parents[1]\nDATA_DIR = ROOT / "data"\nFIG_DIR = ROOT / "figures"\nFIG_DIR.mkdir(exist_ok=True)\n\n# --------------------------------------------------------------------------\n# Constants from the dataset README\n# --------------------------------------------------------------------------\nGOOD_PERIOD_END = "2022-02-28"      # OD flows after this are reduced quality\nSAMPLE_BREAKS = ["2022-01-01", "2022-02-16"]  # abrupt drops in device sample\nANOMALY_DAYS = ["2022-03-08"]       # one-day delivery issue\nNIGHT_HOURS = list(range(0, 6))     # 00:00-05:59 WIB\nDAY_HOURS = list(range(10, 17))     # 10:00-16:59 WIB\nHOLIDAY_WINDOW = ("2021-12-24", "2022-01-02")  # Christmas and New Year\nKAB_ORDER = ["YOGYAKARTA", "SLEMAN", "BANTUL", "KULON PROGO", "GUNUNGKIDUL"]\nDOW_NAMES = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]\n\n\ndef _path(name: str) -> Path:\n    p = DATA_DIR / name\n    if not p.exists():\n        raise FileNotFoundError(\n            f"{p} not found. Run `python scripts/download_data.py` first "\n            "or download the CSV files from Zenodo into the data/ folder."\n        )\n    return p\n\n\n# --------------------------------------------------------------------------\n# Loaders\n# --------------------------------------------------------------------------\ndef load_hourly(drop_anomalies: bool = True) -> pd.DataFrame:\n    """Kapanewon x hour presence (n_maids, n_pings)."""\n    df = pd.read_csv(_path("presence_kapanewon_hourly.csv"),\n                     dtype={"kab_code": str, "kapanewon_code": str})\n    df["date"] = pd.to_datetime(df["date"])\n    if drop_anomalies:\n        df = df[~df["date"].isin(pd.to_datetime(ANOMALY_DAYS))]\n    df["dow"] = df["date"].dt.dayofweek          # 0 = Monday\n    df["is_weekend"] = df["dow"] >= 5\n    df["month"] = df["date"].dt.to_period("M").astype(str)\n    return df\n\n\ndef load_cells(drop_anomalies: bool = True) -> pd.DataFrame:\n    """500 m grid cell x day presence (n_maids)."""\n    df = pd.read_csv(_path("presence_cell500m_daily.csv"))\n    df["date"] = pd.to_datetime(df["date"])\n    if drop_anomalies:\n        df = df[~df["date"].isin(pd.to_datetime(ANOMALY_DAYS))]\n    return df\n\n\ndef load_od() -> pd.DataFrame:\n    """Monthly home-based OD visitor counts, joined with origin denominators."""\n    od = pd.read_csv(_path("od_home_visits_kapanewon_monthly.csv"),\n                     dtype={"origin_code": str, "dest_code": str,\n                            "origin_kab_code": str, "dest_kab_code": str})\n    org = pd.read_csv(_path("od_origin_homes_monthly.csv"),\n                      dtype={"origin_code": str, "origin_kab_code": str})\n    od = od.merge(org[["month", "origin_code", "n_homed_maids",\n                       "n_homed_maids_daytime"]],\n                  on=["month", "origin_code"], how="left")\n    od["share"] = od["n_maids"] / od["n_homed_maids_daytime"]\n    od["is_internal"] = od["origin_code"] == od["dest_code"]\n    od["reduced_quality"] = od["reduced_quality"].astype(str).str.lower() == "true"\n    return od\n\n\ndef load_origins() -> pd.DataFrame:\n    return pd.read_csv(_path("od_origin_homes_monthly.csv"),\n                       dtype={"origin_code": str, "origin_kab_code": str})\n\n\ndef load_home_vs_population() -> pd.DataFrame:\n    return pd.read_csv(_path("home_maids_vs_population_kapanewon.csv"))\n\n\ndef load_night_validation() -> pd.DataFrame:\n    return pd.read_csv(_path("validation_night_presence_kapanewon.csv"))\n\n\n# --------------------------------------------------------------------------\n# Normalisation (README section 6: the device sample shrinks over time,\n# so raw counts must never be compared across dates)\n# --------------------------------------------------------------------------\ndef add_hourly_share(df: pd.DataFrame) -> pd.DataFrame:\n    """Share of each kapanewon in the regional total of the same hour.\n\n    Note: n_maids is a distinct count, so the regional sum is not the exact\n    number of distinct devices. It is used here only as a common denominator\n    that removes the effect of the shrinking sample.\n    """\n    out = df.copy()\n    tot = out.groupby(["date", "hour"])["n_maids"].transform("sum")\n    out["share_hour"] = out["n_maids"] / tot\n    return out\n\n\ndef add_daily_share(cells: pd.DataFrame) -> pd.DataFrame:\n    """Share of each grid cell in the total presence of the same day."""\n    out = cells.copy()\n    out["share_day"] = out["n_maids"] / out.groupby("date")["n_maids"].transform("sum")\n    return out\n\n\ndef hourly_profile(df: pd.DataFrame, value: str = "n_pings") -> pd.DataFrame:\n    """Percentage of each day\'s activity that falls in each hour.\n\n    n_pings is additive, so it is the correct variable for regional totals.\n    Returns one row per date with 24 hour columns.\n    """\n    daily = df.groupby(["date", "hour"])[value].sum().unstack("hour").fillna(0)\n    return daily.div(daily.sum(axis=1), axis=0) * 100\n\n\ndef kapanewon_profiles(df: pd.DataFrame) -> pd.DataFrame:\n    """24-hour shape per kapanewon (each row sums to 1), based on the mean\n    hourly share. Used for functional-zone clustering."""\n    s = add_hourly_share(df)\n    prof = s.groupby(["kapanewon", "hour"])["share_hour"].mean().unstack("hour")\n    prof = prof.reindex(columns=range(24))\n    prof = prof.interpolate(axis=1, limit_direction="both")\n    return prof.div(prof.sum(axis=1), axis=0)\n\n\ndef savefig(fig, name: str, dpi: int = 200) -> Path:\n    path = FIG_DIR / name\n    fig.savefig(path, dpi=dpi, bbox_inches="tight")\n    return path\n')
    import subprocess
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "plotly>=5.24"], check=False)
    import requests
    FILES = {'presence_cell500m_daily.csv': '0b4d544e8c9ab26d69360821e9ebff27', 'presence_kapanewon_hourly.csv': '90a4a8ae6eb68abc68f7a1021a51cce8', 'od_home_visits_kapanewon_monthly.csv': '5857291c608dd5335f6d8833c1ec7b7a', 'od_origin_homes_monthly.csv': 'd14368c48d3937c9146ed9054aee3c79', 'od_summary.csv': '139962d1c8101eb6873be6f902320efb', 'home_maids_vs_population_kapanewon.csv': '0f01f7a3bb7027174cc1b5e1ad0e8224', 'validation_night_presence_kapanewon.csv': '4a8d0f393fd6d4fa17fb0e60c6f7408e', 'tier1_summary.csv': 'd391339521ce3714f1616c430512b0e0'}
    for name, md5 in FILES.items():
        target = f"{BASE}/data/{name}"
        if os.path.exists(target):
            continue
        print("Downloading", name, "...", end=" ", flush=True)
        r = requests.get("https://zenodo.org/records/23157460/files/" + name, params={"download": 1}, timeout=300)
        r.raise_for_status()
        open(target, "wb").write(r.content)
        print("OK" if hashlib.md5(r.content).hexdigest() == md5 else "checksum mismatch")
    os.chdir(f"{BASE}/notebooks")
    print("Setup selesai. Folder kerja:", BASE)


In [ ]:
import sys
sys.path.append("..")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from src import jogjapulse as jp

sns.set_theme(style="whitegrid", context="talk", font_scale=0.8)
od_all = jp.load_od()
od = od_all[~od_all["reduced_quality"]].copy()
print(pd.read_csv(jp.DATA_DIR / "od_summary.csv").T)
print("Months used:", sorted(od["month"].unique()))

## 4.1 Regency-level connection matrix
For each origin regency: of all its residents' (published) daytime visits, what share goes to each destination regency?

In [ ]:
kab = (od.groupby(["month", "origin_kab_kota", "dest_kab_kota"])["n_maids"].sum()
         .groupby(["origin_kab_kota", "dest_kab_kota"]).mean()
         .unstack("dest_kab_kota").reindex(index=jp.KAB_ORDER, columns=jp.KAB_ORDER))
kab_share = kab.div(kab.sum(axis=1), axis=0) * 100

fig, ax = plt.subplots(figsize=(8.5, 6.5))
sns.heatmap(kab_share, annot=True, fmt=".0f", cmap="Blues", ax=ax,
            cbar_kws={"label": "% of origin's visitor-destination pairs"})
ax.set_xlabel("Destination (daytime)")
ax.set_ylabel("Origin (home)")
ax.set_title("Where do residents of each regency go during the day?")
jp.savefig(fig, "04_kab_matrix.png")
plt.show()

## 4.2 Largest inter-kapanewon flows

In [ ]:
inter = od[~od["is_internal"]]
pairs = (inter.groupby(["origin_kapanewon", "dest_kapanewon"])
         .agg(visitors=("n_maids", "mean"), share=("share", "mean"), months=("month", "nunique"))
         .reset_index())
pairs = pairs[pairs["months"] == od["month"].nunique()]          # present in all good months
top_pairs = pairs.nlargest(15, "visitors").copy()
top_pairs["pair"] = top_pairs["origin_kapanewon"].str.title() + " -> " + top_pairs["dest_kapanewon"].str.title()

fig, ax = plt.subplots(figsize=(10, 7))
ax.barh(top_pairs["pair"][::-1], top_pairs["visitors"][::-1], color="#1f4e79")
for y, (v, sh) in enumerate(zip(top_pairs["visitors"][::-1], top_pairs["share"][::-1])):
    ax.text(v, y, f"  {sh*100:.0f}% of origin", va="center", fontsize=8)
ax.set_xlabel("Mean distinct visitors per month")
ax.set_title("Top 15 home -> daytime destination flows (Dec 2021 - Feb 2022)")
jp.savefig(fig, "04_top_flows.png")
plt.show()

## 4.3 Attraction: which kapanewon draw visitors from the most places?

In [ ]:
att = (inter.groupby(["month", "dest_kab_kota", "dest_kapanewon"])
       .agg(n_origins=("origin_code", "nunique"), visitors=("n_maids", "sum"))
       .groupby(["dest_kab_kota", "dest_kapanewon"]).mean().reset_index())
top_att = att.nlargest(12, "visitors")
fig, ax = plt.subplots(figsize=(10, 6))
sns.scatterplot(data=att, x="n_origins", y="visitors", hue="dest_kab_kota", s=70, ax=ax, alpha=0.8)
for _, r in top_att.iterrows():
    ax.annotate(r["dest_kapanewon"].title(), (r["n_origins"], r["visitors"]), fontsize=8,
                xytext=(3, 3), textcoords="offset points")
ax.set_xlabel("Number of origin kapanewon sending visitors (per month)")
ax.set_ylabel("Visitors from other kapanewon (per month)")
ax.set_title("Attraction of each kapanewon")
ax.legend(title="", fontsize=8)
jp.savefig(fig, "04_attraction.png")
plt.show()

## Key findings (fill in after running)
- Sleman is the hub: it receives the largest share of cross-regency visits (e.g. 30% of Bantul residents' visits to other kapanewon go to Sleman, more than to the city).
- Strongest flows link the northern Sleman cluster (Ngaglik, Depok, Ngemplak, Pakem) and Banguntapan -> Umbulharjo.
- Depok attracts visitors from the most origins (73 kapanewon).

In [ ]:
# @title Unduh semua grafik (Colab)
# Membuat figures.zip lalu mengunduhnya ke laptop, untuk slide dan untuk di-push ke GitHub.
if IN_COLAB:
    import shutil
    from google.colab import files as colab_files
    shutil.make_archive("/content/figures", "zip", "/content/jogjapulse/figures")
    colab_files.download("/content/figures.zip")
